# 02 — Baselines

Este notebook estabelece os modelos de referência para o projeto.

Serão avaliadas três abordagens:

1. Baseline de classe majoritária;
2. TF-IDF + Regressão Logística utilizando holdout;
3. TF-IDF + Regressão Logística utilizando validação cruzada.

O objetivo dos baselines é estabelecer referências quantitativas antes da realização de tuning de hiperparâmetros e da comparação com outros modelos.

As principais métricas utilizadas serão:

- Accuracy;
- F1 Macro.

A utilização do F1 Macro é especialmente relevante quando as classes apresentam distribuições diferentes.

In [ ]:
#Alteração: centralizamos os imports e configurações compartilhadas pelos experimentos de baseline.

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
from sklearn.model_selection import (
    train_test_split,
    cross_validate,
    StratifiedKFold,
)
from sklearn.pipeline import Pipeline

import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.config import (
    CLASSES,
    RANDOM_STATE,
    TEST_SIZE,
    N_SPLITS,
)

from src.data import carregar_dados_treino
from registro_resultados import registrar_resultado

In [ ]:
#Alteração: centralizamos os imports e configurações compartilhadas pelos experimentos de baseline.

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
from sklearn.model_selection import (
    train_test_split,
    cross_validate,
    StratifiedKFold,
)
from sklearn.pipeline import Pipeline

import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.config import (
    CLASSES,
    RANDOM_STATE,
    TEST_SIZE,
    N_SPLITS,
)

from src.data import carregar_dados_treino
from registro_resultados import registrar_resultado

## 1. Baseline de Classe Majoritária

O primeiro baseline não utiliza nenhuma informação textual.

O modelo simplesmente identifica a classe mais frequente no conjunto de treinamento e atribui essa classe a todas as amostras.

Esse experimento estabelece uma referência mínima.

Para que um modelo de PLN seja considerado informativo, esperamos que ele apresente desempenho superior a essa referência, especialmente em métricas como F1 Macro.

In [ ]:
# Alteração: adicionamos a divisão estratificada para avaliar o baseline de classe majoritária nas mesmas condições do baseline principal.

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Tamanho do treino:", len(X_train))
print("Tamanho da validação:", len(X_val))

In [ ]:
# Alteração: implementamos corretamente o baseline de classe majoritária utilizando DummyClassifier.

modelo_majoritario = DummyClassifier(
    strategy="most_frequent"
)

modelo_majoritario.fit(X_train, y_train)

y_pred_majoritario = modelo_majoritario.predict(X_val)

In [ ]:
# Alteração: calculamos Accuracy e F1 Macro para estabelecer a referência mínima do problema.

accuracy_majoritario = accuracy_score(
    y_val,
    y_pred_majoritario
)

f1_majoritario = f1_score(
    y_val,
    y_pred_majoritario,
    average="macro",
    zero_division=0
)

classe_majoritaria = (
    modelo_majoritario.classes_[
        modelo_majoritario.class_prior_.argmax()
    ]
)

print("Classe majoritária:", classe_majoritaria)
print(f"Accuracy: {accuracy_majoritario:.4f}")
print(f"F1 Macro: {f1_majoritario:.4f}")

In [ ]:
# Alteração: registramos o baseline de classe majoritária na tabela consolidada de experimentos.

resultado_majoritario = {
    "experimento": "Classe Majoritária",
    "representacao": "Nenhuma",
    "modelo": "DummyClassifier",
    "acuracia": accuracy_majoritario,
    "f1_macro": f1_majoritario,
}

registrar_resultado(resultado_majoritario)

## 2. Baseline TF-IDF + Regressão Logística

Neste experimento utilizamos uma representação textual baseada em TF-IDF e um classificador de Regressão Logística.

O TF-IDF transforma os documentos em uma representação numérica baseada na importância relativa dos termos.

O modelo é colocado dentro de um Pipeline para garantir que o vocabulário e os pesos TF-IDF sejam aprendidos exclusivamente a partir dos dados de treinamento.

Essa organização evita data leakage e será mantida nos experimentos posteriores.

In [ ]:
# Alteração: criamos o pipeline baseline com TF-IDF e Regressão Logística, mantendo a transformação textual dentro do processo de treinamento.

pipeline_baseline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer()
    ),
    (
        "logistic",
        LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE
        )
    ),
])

In [ ]:
# Alteração: treinamos o pipeline utilizando somente a partição de treinamento criada anteriormente.

pipeline_baseline.fit(
    X_train,
    y_train
)

In [ ]:
# Alteração: geramos as previsões sobre o conjunto de validação para permitir a avaliação do baseline textual.

y_pred_baseline = pipeline_baseline.predict(
    X_val
)

In [ ]:
# Alteração: calculamos as métricas principais do baseline textual utilizando Accuracy e F1 Macro.

accuracy_baseline = accuracy_score(
    y_val,
    y_pred_baseline
)

f1_baseline = f1_score(
    y_val,
    y_pred_baseline,
    average="macro",
    zero_division=0
)

print("RESULTADO DO BASELINE")
print("=====================")
print(f"Accuracy: {accuracy_baseline:.4f}")
print(f"F1 Macro: {f1_baseline:.4f}")

In [ ]:
# Alteração: adicionamos a matriz de confusão para analisar como o baseline se comporta individualmente entre as classes.

matriz_baseline = confusion_matrix(
    y_val,
    y_pred_baseline,
    labels=CLASSES
)

matriz_baseline_df = pd.DataFrame(
    matriz_baseline,
    index=[f"Real {classe}" for classe in CLASSES],
    columns=[f"Predito {classe}" for classe in CLASSES]
)

matriz_baseline_df

In [ ]:
# Alteração: detalhamos Precision, Recall e F1 de cada classe para complementar as métricas agregadas.

relatorio_baseline = classification_report(
    y_val,
    y_pred_baseline,
    labels=CLASSES,
    target_names=CLASSES,
    digits=4,
    zero_division=0,
    output_dict=True
)

relatorio_baseline_df = pd.DataFrame(
    relatorio_baseline
).T

relatorio_baseline_df

In [ ]:
# Alteração: registramos o desempenho do baseline TF-IDF + Regressão Logística na tabela consolidada de experimentos.

resultado_baseline = {
    "experimento": "Holdout - Baseline",
    "representacao": "TF-IDF",
    "modelo": "Regressão Logística",
    "acuracia": accuracy_baseline,
    "f1_macro": f1_baseline,
}

registrar_resultado(resultado_baseline)

## 3. Baseline com Validação Cruzada

O experimento anterior utilizou uma única divisão entre treinamento e validação.

Agora utilizaremos validação cruzada estratificada para avaliar o mesmo pipeline em diferentes partições dos dados.

A estratificação busca preservar a distribuição das classes em cada fold.

O resultado será apresentado como média ± desvio padrão, permitindo observar não apenas o desempenho médio, mas também sua variação entre as partições.

In [ ]:
# Alteração: criamos uma estratégia de validação cruzada estratificada utilizando a configuração centralizada do projeto.

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

print(f"Número de folds: {N_SPLITS}")

In [ ]:
# Alteração: criamos um novo pipeline para que cada fold aprenda seu próprio vocabulário TF-IDF sem vazamento entre as partições.

pipeline_cv = Pipeline([
    (
        "tfidf",
        TfidfVectorizer()
    ),
    (
        "logistic",
        LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE
        )
    ),
])

In [ ]:
# Alteração: executamos a validação cruzada utilizando Accuracy e F1 Macro simultaneamente.

resultados_cv = cross_validate(
    pipeline_cv,
    X,
    y,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "f1_macro": "f1_macro",
    },
    return_train_score=True,
    n_jobs=-1
)

resultados_cv_df = pd.DataFrame({
    "fold": range(1, N_SPLITS + 1),
    "accuracy_treino": resultados_cv["train_accuracy"],
    "accuracy_validacao": resultados_cv["test_accuracy"],
    "f1_macro_treino": resultados_cv["train_f1_macro"],
    "f1_macro_validacao": resultados_cv["test_f1_macro"],
})

resultados_cv_df

In [ ]:
# Alteração: resumimos os resultados da validação cruzada como média e desvio padrão entre os folds.

resumo_cv = {
    "acuracia_media": resultados_cv["test_accuracy"].mean(),
    "desvio_acuracia": resultados_cv["test_accuracy"].std(),
    "f1_macro_medio": resultados_cv["test_f1_macro"].mean(),
    "desvio_f1_macro": resultados_cv["test_f1_macro"].std(),
    "acuracia_treino_media": resultados_cv["train_accuracy"].mean(),
    "f1_macro_treino_medio": resultados_cv["train_f1_macro"].mean(),
}

print(
    f"Accuracy: "
    f"{resumo_cv['acuracia_media']:.4f} "
    f"± {resumo_cv['desvio_acuracia']:.4f}"
)

print(
    f"F1 Macro: "
    f"{resumo_cv['f1_macro_medio']:.4f} "
    f"± {resumo_cv['desvio_f1_macro']:.4f}"
)

print(
    f"Accuracy treino: "
    f"{resumo_cv['acuracia_treino_media']:.4f}"
)

print(
    f"F1 Macro treino: "
    f"{resumo_cv['f1_macro_treino_medio']:.4f}"
)

In [ ]:
# Alteração: calculamos o gap entre treinamento e validação para observar possíveis sinais de sobreajuste.

gap_accuracy = (
    resumo_cv["acuracia_treino_media"]
    - resumo_cv["acuracia_media"]
)

gap_f1 = (
    resumo_cv["f1_macro_treino_medio"]
    - resumo_cv["f1_macro_medio"]
)

print(f"Gap Accuracy: {gap_accuracy:.4f}")
print(f"Gap F1 Macro: {gap_f1:.4f}")

## 4. Comparação dos Baselines

Nesta etapa reunimos os resultados obtidos pelos experimentos iniciais.

O objetivo não é selecionar definitivamente o melhor modelo, mas estabelecer as referências que serão utilizadas nas etapas seguintes.

A comparação deve considerar principalmente:

- desempenho em relação ao baseline de classe majoritária;
- Accuracy;
- F1 Macro;
- diferença entre treinamento e validação;
- estabilidade observada entre os folds da validação cruzada.

Os experimentos de tuning e comparação de modelos serão realizados posteriormente.

In [ ]:
# Alteração: carregamos a tabela consolidada para visualizar conjuntamente os resultados registrados pelos experimentos.

ARQUIVO_RESULTADOS = "results/resultados_experimentos.csv"

resultados_experimentos = pd.read_csv(
    ARQUIVO_RESULTADOS
)

resultados_experimentos

In [ ]:
# Alteração: selecionamos as principais métricas disponíveis para produzir uma comparação compacta entre os baselines.

colunas_comparacao = [
    coluna
    for coluna in [
        "experimento",
        "representacao",
        "modelo",
        "acuracia",
        "acuracia_media",
        "desvio_acuracia",
        "f1_macro",
        "f1_macro_medio",
        "desvio_f1_macro",
    ]
    if coluna in resultados_experimentos.columns
]

resultados_experimentos[colunas_comparacao]

## Conclusões dos Baselines

Os experimentos realizados nesta etapa estabelecem as referências iniciais para o restante do projeto.

O baseline de classe majoritária permite verificar o desempenho obtido sem utilizar informação textual.

O baseline TF-IDF + Regressão Logística permite verificar o ganho obtido quando o conteúdo dos documentos é utilizado para a classificação.

A validação cruzada complementa o experimento holdout ao avaliar o pipeline em múltiplas partições dos dados, fornecendo uma estimativa de média e variabilidade do desempenho.

Esses resultados serão utilizados como referência para o próximo notebook, dedicado ao tuning dos modelos e hiperparâmetros.